# Matrix Reconstruction (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [10]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Step 1: Load the desired dataset (test.pt / all.pt)
Load correlation matrices.

In [11]:
FILE_NAME = 'data_00_20'
DATASET_NAME = f'{FILE_NAME}_w724_s10'
RUN = 'VAE_20dim_cholesky_06_loss'
RESULTS_JSON_PATH = f'models/{DATASET_NAME}/VAE/{RUN}/run_results.json'
DATASET = 'train' #'test' or 'all'


if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / FILE_NAME / 'dataset'

dataset_dir = base_dir / DATASET_NAME

if DATASET == 'all':
    MATRIX_FILE = dataset_dir / 'all.pt'
elif DATASET == 'test':
    MATRIX_FILE = dataset_dir / 'test.pt'
elif DATASET == 'train':
    MATRIX_FILE = dataset_dir / 'train.pt'

if not MATRIX_FILE.exists():
    raise FileNotFoundError(
        f"File '{MATRIX_FILE.name}' not found in: {dataset_dir.absolute()}"
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Test Matrix: {MATRIX_FILE.name}')

Environment: Local PC
Dataset selected: data_00_20_w724_s10
Test Matrix: train.pt


In [12]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        chol_tensor = payload.get('chol_tensor', None) # AGGIUNTO
        indices = payload.get('indices', None)
        meta = {k: v for k, v in payload.items() if k not in ('corr_tensor', 'chol_tensor')}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        chol_tensor = None
        indices = None
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
        
    if chol_tensor is None:
        raise KeyError('chol_tensor key not found in .pt file. Usa il dataset aggiornato.')

    # Ritorniamo i 4 elementi
    return corr_tensor.float(), chol_tensor.float(), indices, meta

# --- FIX: Definisci il dizionario DATASET_FILES ---
# Mettiamo in corrispondenza il nome del dataset con il suo path
DATASET_FILES = {DATASET: MATRIX_FILE}

SAMPLE_DATASET = next(iter(DATASET_FILES))

# --- MODIFICA QUI: Aggiunto sample_chol ---
sample_corr, sample_chol, sample_indices, sample_meta = load_corr_payload(DATASET_FILES[SAMPLE_DATASET])

print(f'Sample dataset: {SAMPLE_DATASET}')
print(f'Correlation tensor shape: {sample_corr.shape}')
print(f'Cholesky tensor shape: {sample_chol.shape}') # Utile per verificare!

if sample_indices is not None:
    print(f'Indices: {sample_indices[:10]}')
else:
    print('Indices: not found in payload')

Sample dataset: train
Correlation tensor shape: torch.Size([288, 362, 362])
Cholesky tensor shape: torch.Size([288, 362, 362])
Indices: [203, 369, 213, 185, 124, 96, 245, 171, 256, 179]


## Step 2: Prepare Matrices (Full Dataset)
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.

In [13]:
def prepare_inputs(chol_tensor: torch.Tensor):
    # Lavoriamo con Cholesky!
    all_chol_np = chol_tensor.numpy().astype(np.float32)
    n_matrices, n_assets, _ = all_chol_np.shape

    # Formula per Cholesky con diagonale
    n_features = n_assets * (n_assets + 1) // 2
    
    # k=0 per includere la diagonale
    tril_idx = np.tril_indices(n_assets, k=0)
    
    extracted_np = all_chol_np[:, tril_idx[0], tril_idx[1]]
    x_all = torch.from_numpy(extracted_np)

    return all_chol_np, x_all, n_assets, n_features, tril_idx

sample_all_np, sample_x_all, N_ASSETS, N_FEATURES, SAMPLE_TRIL_IDX = prepare_inputs(sample_chol)

print(f"{'='*40}")
print(f"Number of matrices   : {sample_all_np.shape[0]}")
print(f"Original matrix shape: ({N_ASSETS}, {N_ASSETS})")
print(f"Flattened input size : {N_FEATURES}")
print(f"Sample tensor shape  : {tuple(sample_x_all.shape)}")
print(f"{'='*40}")

Number of matrices   : 288
Original matrix shape: (362, 362)
Flattened input size : 65703
Sample tensor shape  : (288, 65703)


## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [14]:
class VAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], latent_dim: int):
        super().__init__()

        # Encoder
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers.append(nn.Linear(prev, h))
            enc_layers.append(nn.LeakyReLU(0.01))
            prev = h
        self.encoder = nn.Sequential(*enc_layers)

        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        # Decoder
        dec_layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers.append(nn.Linear(prev, h))
            dec_layers.append(nn.LeakyReLU(0.01))
            prev = h
        dec_layers.append(nn.Linear(prev, input_dim))
        #dec_layers.append(nn.Tanh())
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x: torch.Tensor):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        sample = mu + (eps * std)
        return sample

    def decode(self, z: torch.Tensor):
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        if self.training:
            # Durante l'addestramento (model.train()), usa il campionamento stocastico
            z = self.reparameterize(mu, logvar)
        else:
            # Durante la validazione/test (model.eval()), usa l'aspettativa esatta
            z = mu
        x_hat = self.decode(z)
        return x_hat, mu, logvar

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [15]:
def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p

# --- 1. RISOLUZIONE E CARICAMENTO DEL JSON ---
results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

OUTPUT_DIR_NAME = 'analysis_outputs'
analysis_dir = results_path.parent / OUTPUT_DIR_NAME
analysis_dir.mkdir(parents=True, exist_ok=True)

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

# --- 2. ESTRAZIONE PARAMETRI VAE ---
model_cfg = results.get('model', {})

latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
if hidden_dims is None:
    raise ValueError('hidden_dims missing for VAE in results JSON')

input_dim = int(model_cfg.get('input_dim', sample_x_all.shape[1]))
if input_dim != sample_x_all.shape[1]:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={sample_x_all.shape[1]}')

# --- 3. INIZIALIZZAZIONE DEL MODELLO VAE ---
model = VAE(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dims=hidden_dims,
).to(device)

# --- 4. CARICAMENTO DEI PESI ---
weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    weights_path = f'best_model.pt' # Fallback standard per come abbiamo salvato il VAE in precedenza

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)

if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint

model.load_state_dict(state_dict)
model.eval()

# --- INIZIO AGGIUNTA: Caricamento statistiche di normalizzazione ---
stats_dir = weights_path.parent
train_mean_path = stats_dir / 'train_mean.pt'
train_std_path = stats_dir / 'train_std.pt'

if not train_mean_path.exists() or not train_std_path.exists():
    raise FileNotFoundError(f"File di normalizzazione mancanti in: {stats_dir}. Assicurati di averli copiati.")

# Carichiamo i tensori e assicuriamoci che siano sulla CPU
train_mean = torch.load(train_mean_path, map_location='cpu')
train_std = torch.load(train_std_path, map_location='cpu')

# Creiamo subito anche le copie NumPy per velocizzare la denormalizzazione
train_mean_np = train_mean.numpy()
train_std_np = train_std.numpy()
# --- FINE AGGIUNTA ---

print(f'Model type: VAE | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')
print(f'Loaded norm stats: {stats_dir.name}')

Model type: VAE | latent_dim=20 | input_dim=65703
Loaded weights: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\VAE\VAE_20dim_cholesky_06_loss\best_model.pt
Loaded norm stats: VAE_20dim_cholesky_06_loss


In [16]:
print(model)

VAE(
  (encoder): Sequential(
    (0): Linear(in_features=65703, out_features=2048, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=2048, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
  )
  (fc_mu): Linear(in_features=512, out_features=20, bias=True)
  (fc_logvar): Linear(in_features=512, out_features=20, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=20, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=2048, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=2048, out_features=65703, bias=True)
  )
)


## Step 5: Latent Space Analysis + Reconstruction Errors Analysis
Encode the matrices into the latent space and analyze feature distributions + Analyse MSE, MAE and Frobenius.

In [17]:
# 1. Funzione di ricostruzione con gestione automatica del device
def compute_reconstruction(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 64, device=None):
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []
    reconstructions = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            
            # --- MODIFICHE VAE ---
            # 1. Chiamiamo model.encode() invece di model.encoder() per ottenere Media e Log-Varianza
            mu, logvar = model.encode(xb)
            
            # 2. Usiamo 'mu' (la media) come nostro spazio latente deterministico per i grafici/analisi
            latents.append(mu.cpu().numpy())
            
            # 3. Decodifichiamo direttamente 'mu', bypassando il rumore casuale di reparameterize()
            recon = model.decode(mu)
            reconstructions.append(recon.cpu().numpy())
            # ---------------------

    return np.concatenate(latents, axis=0), np.concatenate(reconstructions, axis=0)


# 2. Funzione per gli errori (rimane invariata, gestisce matrici 2D piatte o 3D quadrate)
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    if original.shape != reconstructed.shape:
        raise ValueError(f'Input shapes mismatch: {original.shape} vs {reconstructed.shape}')

    diff = original - reconstructed

    
    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [18]:
# ==========================================
# ESECUZIONE DEL CALCOLO E RICOSTRUZIONE
# ==========================================

# Convertiamo la matrice di correlazione originale (Ground Truth) in NumPy per il calcolo errori
all_corr_np = sample_corr.numpy().astype(np.float32)

# --- MODIFICA 1: NORMALIZZAZIONE (Z-Score) ---
# Usiamo sample_x_all (il tensore estratto in precedenza)
x_all_norm = (sample_x_all - train_mean) / train_std

# Passiamo x_all_norm al modello
latents_all, recon_flat_norm = compute_reconstruction(model, x_all_norm, batch_size=64)

# --- MODIFICA 2: DENORMALIZZAZIONE ---
recon_flat = (recon_flat_norm * train_std_np) + train_mean_np

# --- IL FIX FONDAMENTALE: Ricostruzione Geometrica (Cholesky) ---
n_matrices, n_assets, _ = all_corr_np.shape

# 1. Inizializziamo le matrici vuote per il fattore L
recon_L = np.zeros((n_matrices, n_assets, n_assets), dtype=np.float32)

# 2. Estraiamo indici del triangolo inferiore INCLUSA la diagonale (k=0)
tril_idx = np.tril_indices(n_assets, k=0)

# 3. Riempiamo la parte triangolare inferiore
recon_L[:, tril_idx[0], tril_idx[1]] = recon_flat

# 4. Calcolo PSD: C = L @ L.T
recon_PSD = recon_L @ np.transpose(recon_L, axes=(0, 2, 1))

# 5. Normalizzazione a matrice di correlazione
diag = np.diagonal(recon_PSD, axis1=1, axis2=2)
d_inv_sqrt = 1.0 / np.sqrt(np.maximum(diag, 1e-12))

# Vettorializzazione del prodotto D^{-1/2} * C * D^{-1/2}
recon_all = recon_PSD * d_inv_sqrt[:, :, np.newaxis] * d_inv_sqrt[:, np.newaxis, :]

# 6. Clip di sicurezza finale e ripristino diagonale esatta
recon_all = np.clip(recon_all, -1.0, 1.0)
diag_idx = np.arange(n_assets)
recon_all[:, diag_idx, diag_idx] = 1.0
# -----------------------------------------------------

# Ora calcoliamo gli errori confrontando con la Correlazione Originale!
errors_df, summary_df = reconstruction_errors(all_corr_np, recon_all)

# ==========================================
# GESTIONE DATAFRAME E SALVATAGGI
# ==========================================

latent_dim = latents_all.shape[1]
latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
latent_df = pd.DataFrame(latents_all, columns=latent_cols)

# Gestione sicura degli indici
if sample_indices is None:
    sample_indices = np.arange(len(errors_df))

if len(sample_indices) != len(errors_df):
    raise ValueError(f'Error rows ({len(errors_df)}) do not match indices ({len(sample_indices)})')
errors_df.insert(0, 'matrix_idx', sample_indices)

print(f'Reconstruction error summary ({DATASET} data):')
display(summary_df)

errors_json_path = analysis_dir / f'reconstruction_errors_{DATASET}_{RUN}.json'
summary_json_path = analysis_dir / f'reconstruction_summary_{DATASET}_{RUN}.json'
errors_df.to_json(errors_json_path, orient='records', indent=2)
summary_df.to_json(summary_json_path, orient='records', indent=2)

print(f'Saved per-matrix errors: {errors_json_path}')
print(f'Saved summary stats: {summary_json_path}')

# Load original payload and create extended version with reconstructed matrices
original_payload = torch.load(MATRIX_FILE, map_location='cpu')
if isinstance(original_payload, dict):
    extended_payload = original_payload.copy()
else:
    extended_payload = {'corr_tensor': original_payload}

# Add reconstructed matrices as a new key
recon_tensor = torch.from_numpy(recon_all).float()
extended_payload['corr_tensor_reconstructed'] = recon_tensor

# Aggiunto: Salvataggio anche dei fattori L ricostruiti
recon_L_tensor = torch.from_numpy(recon_L).float()
extended_payload['chol_tensor_reconstructed'] = recon_L_tensor

# Estrazione sicura dei tickers da sample_meta
tickers = None
if isinstance(sample_meta, dict):
    tickers = sample_meta.get('meta', {}).get('tickers', None)
    if tickers is None:
        tickers = sample_meta.get('tickers', None)

if tickers is not None:
    extended_payload['tickers'] = tickers

# Save to file with the same name but in original location
output_path = analysis_dir / f'{DATASET}_reconstructed_{RUN}.pt'
torch.save(extended_payload, output_path)
print(f'Saved reconstructed matrices: {output_path}')

print(f'\nLatent distribution summary ({DATASET} data):')
display(latent_df.describe().T)

valid_cols = [col for col in latent_cols if latent_df[col].notna().any() and latent_df[col].nunique() > 1]
latent_df = latent_df[valid_cols]

if len(sample_indices) != len(latent_df):
    raise ValueError(f'Latent rows ({len(latent_df)}) do not match indices ({len(sample_indices)})')
latent_df.insert(0, 'matrix_idx', sample_indices)

latent_json_path = analysis_dir / f'latent_{DATASET}_{RUN}.json'
latent_df.to_json(latent_json_path, orient='records', indent=2)
print(f'Saved latent samples: {latent_json_path}')

if len(valid_cols) < 2:
    print('Not enough valid latent dimensions for pairwise plots.')
elif len(valid_cols) >= 20:
    print(f'Skipping pairwise plot: too many latent dimensions ({len(valid_cols)} > 20).')
else:
    plot_df = latent_df[valid_cols]
    grid = sns.PairGrid(plot_df, corner=True, diag_sharey=False)
    grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
    grid.figure.suptitle(f'Pairwise latent dimension plots ({DATASET} data)', y=1.02)
    pairplot_path = analysis_dir / f'latent_pairwise_{DATASET}_{RUN}.png'
    grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved latent pairwise plot: {pairplot_path}')

Reconstruction error summary (train data):


,mean,std,min,median,max
MSE,0.000087,0.000077,0.000014,0.000065,0.000762
MAE,0.006436,0.002664,0.002860,0.005995,0.021751
Frobenius,3.162532,1.177206,1.358025,2.908820,9.993035


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\VAE\VAE_20dim_cholesky_06_loss\analysis_outputs\reconstruction_errors_train_VAE_20dim_cholesky_06_loss.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\VAE\VAE_20dim_cholesky_06_loss\analysis_outputs\reconstruction_summary_train_VAE_20dim_cholesky_06_loss.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\VAE\VAE_20dim_cholesky_06_loss\analysis_outputs\train_reconstructed_VAE_20dim_cholesky_06_loss.pt

Latent distribution summary (train data):


,count,mean,std,min,25%,50%,75%,max
z1,288.0,-0.022029,0.952871,-1.540715,-0.825198,-0.168413,0.727625,2.176715
z2,288.0,-0.010640,0.121237,-0.417052,-0.053825,-0.013489,0.050464,0.473967
z3,288.0,0.006630,0.045562,-0.110277,-0.023201,-0.000492,0.029871,0.129991
z4,288.0,0.022632,0.880824,-2.785981,-0.355345,-0.008212,0.352114,2.140585
z5,288.0,-0.013750,0.835182,-2.281355,-0.126259,0.094143,0.250831,2.727740
z6,288.0,0.006428,0.600803,-2.230914,-0.078720,0.009272,0.130072,1.995008
z7,288.0,0.005520,0.992229,-2.503599,-0.583565,-0.138897,0.571592,2.391411
z8,288.0,-0.002816,0.058885,-0.123124,-0.036729,-0.002243,0.023921,0.235274
z9,288.0,-0.015634,1.033382,-2.388036,-0.524634,0.003740,0.516627,2.643983
z10,288.0,0.014352,0.566513,-3.087365,-0.008176,0.068042,0.198835,1.149476


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\VAE\VAE_20dim_cholesky_06_loss\analysis_outputs\latent_train_VAE_20dim_cholesky_06_loss.json
Skipping pairwise plot: too many latent dimensions (20 > 20).
